# Service Cockpit — Attention Logic

This notebook translates the exploratory findings from `01_eda.ipynb` into a small, explicit prioritization layer for the prototype.

The goal is not to diagnose heat-pump faults.

The goal is to decide which units deserve attention, why they were surfaced, and whether the next action belongs to:

- service / technician review
- data / connectivity review
- no immediate action

The logic should remain explainable and auditable.

### Principles

- preserve raw OEM signals
- separate equipment-related signals from data-health signals
- avoid one synthetic health score
- allow multiple simultaneous attention signals per unit
- do not interpret undocumented OEM codes as confirmed diagnoses

## 1. Load cleaned analytical inputs

This notebook starts from the canonical outputs of the EDA rather than repeating identifier reconciliation and source-data cleaning.

This keeps exploratory decisions separate from the prioritization logic used by the product.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

OUTPUT_DIR = Path("../outputs")

unit_base = pd.read_csv(
    OUTPUT_DIR / "unit_base.csv"
)

units = pd.read_csv(
    OUTPUT_DIR / "units_clean.csv"
)

telemetry = pd.read_csv(
    OUTPUT_DIR / "telemetry_clean.csv"
)

unit_error_recurrence = pd.read_csv(
    OUTPUT_DIR / "unit_error_recurrence.csv"
)

error_code_summary = pd.read_csv(
    OUTPUT_DIR / "error_code_summary.csv"
)

### 1.1 Restore datetime fields

CSV files do not preserve pandas datetime types, so date columns are parsed again before prioritization logic is applied.

In [2]:
unit_date_columns = [
    "commissioning_date",
    "last_service_visit",
    "first_reading",
    "latest_reading"
]

for column in unit_date_columns:
    if column in units.columns:
        units[column] = pd.to_datetime(
            units[column],
            errors="coerce",
            format="mixed",
            dayfirst=True
        )

for column in [
    "commissioning_date",
    "last_service_visit"
]:
    if column in unit_base.columns:
        unit_base[column] = pd.to_datetime(
            unit_base[column],
            errors="coerce",
            format="mixed",
            dayfirst=True
        )

telemetry["reading_date"] = pd.to_datetime(
    telemetry["reading_date"],
    errors="coerce"
)

for df in [
    unit_error_recurrence,
    error_code_summary
]:
    for column in [
        "first_seen",
        "last_seen"
    ]:
        if column in df.columns:
            df[column] = pd.to_datetime(
                df[column],
                errors="coerce"
            )

### 1.2 Validate analytical inputs

Before defining attention logic, verify that the exported datasets preserve the main EDA results.

In [3]:
print("Canonical units:", units["unit_id"].nunique())
print(
    "Units with telemetry:",
    units["has_telemetry"].sum()
)

print(
    "Telemetry rows:",
    len(telemetry)
)

print(
    "Unit-level error patterns:",
    len(unit_error_recurrence)
)

Canonical units: 400
Units with telemetry: 268
Telemetry rows: 15457
Unit-level error patterns: 473


## 2. Define the attention taxonomy

The EDA identified two fundamentally different reasons a unit may require attention.

### A. Equipment-signal attention

The unit continues reporting but emits an unusual, persistent OEM-specific signal.

Current strongest example:

- recurrent OEM C `ALM_HP_LOWFLOW`

This is an observed raw signal, not a confirmed technical diagnosis.

### B. Data-health attention

The cockpit cannot reliably determine the current state of the unit.

Current strongest example:

- telemetry previously existed but stopped before the end of the dataset

### C. Data limitation

The unit is visible, but diagnostic depth is reduced because:

- the OEM does not expose a field
- a field is missing
- a value is suspected to be invalid

These conditions may appear alongside equipment- or data-health signals.

### Routing principle

The worklist should distinguish between:

- **technician review**
- **data / connectivity review**

rather than placing every anomaly into one generic critical queue.

## 3. Define high-confidence candidate signals

The first version uses only signals that the EDA supports clearly.

### Signal 1 — telemetry stopped

A reporting unit is flagged when its latest reading is at least two days older than the end of the extract.

**🟡 Assumption:** two or more days without telemetry is unusual enough in this dataset to justify review.

### Signal 2 — recurrent uncommon raw error

A raw error signal is surfaced when:

- it appears on at least two days for the same unit
- it affects fewer than 20% of reporting units for that OEM
- it was observed within the final seven days of the dataset

**🟡 Assumption:** OEM signals affecting fewer than 20% of the reporting fleet are more useful for prioritization than very common signals.

**🟡 Assumption:** a signal seen within the final seven days is relevant enough for a next-day review queue.

In [4]:
dataset_end = telemetry["reading_date"].max()

attention = units[
    [
        "unit_id",
        "vendor_canonical",
        "connectivity",
        "latest_reading",
        "days_since_last_reading",
        "telemetry_coverage_pct"
    ]
].copy()

attention["signal_stale_telemetry"] = (
    attention["days_since_last_reading"] >= 2
)

In [5]:
reporting_units_by_vendor = (
    units[
        units["has_telemetry"]
    ]
    .groupby("vendor_canonical")["unit_id"]
    .nunique()
)

error_code_summary["reporting_units"] = (
    error_code_summary["vendor_canonical"]
    .map(reporting_units_by_vendor)
)

error_code_summary["affected_unit_pct"] = (
    error_code_summary["affected_units"]
    / error_code_summary["reporting_units"]
    * 100
).round(1)

In [6]:
error_context = unit_error_recurrence.merge(
    error_code_summary[
        [
            "vendor_canonical",
            "error_code_raw",
            "affected_unit_pct"
        ]
    ],
    on=[
        "vendor_canonical",
        "error_code_raw"
    ],
    how="left"
)

error_context["days_since_last_error"] = (
    dataset_end
    - error_context["last_seen"]
).dt.days

discriminating_errors = error_context[
    (error_context["days_with_code"] >= 2)
    & (error_context["affected_unit_pct"] < 20)
    & (error_context["days_since_last_error"] <= 7)
].copy()

In [7]:
equipment_signals = (
    discriminating_errors
    .groupby(
        [
            "vendor_canonical",
            "unit_id"
        ]
    )
    .agg(
        recurrent_error_days=("days_with_code", "max"),
        latest_error=("last_seen", "max"),
        raw_error_signal=(
            "error_code_raw",
            lambda x: ", ".join(
                sorted(set(map(str, x)))
            )
        )
    )
    .reset_index()
)

attention = attention.merge(
    equipment_signals,
    on=[
        "vendor_canonical",
        "unit_id"
    ],
    how="left"
)

attention["signal_recurrent_error"] = (
    attention["raw_error_signal"].notna()
)

In [8]:
display(
    attention[
        attention[
            [
                "signal_stale_telemetry",
                "signal_recurrent_error"
            ]
        ].any(axis=1)
    ][
        [
            "unit_id",
            "vendor_canonical",
            "signal_stale_telemetry",
            "signal_recurrent_error",
            "raw_error_signal",
            "recurrent_error_days",
            "latest_error",
            "latest_reading"
        ]
    ]
)

,unit_id,vendor_canonical,signal_stale_telemetry,signal_recurrent_error,raw_error_signal,recurrent_error_days,latest_error,latest_reading
22,TH-02023,A,True,False,NaN,NaN,NaT,2026-07-20
279,TH-02280,B,True,False,NaN,NaN,NaT,2026-07-20
297,TH-02298,C,False,True,ALM_HP_LOWFLOW,14.0,2026-07-30,2026-07-30
303,TH-02304,C,True,False,NaN,NaN,NaT,2026-07-20
311,TH-02312,C,False,True,ALM_HP_LOWFLOW,14.0,2026-07-30,2026-07-30
394,TH-02395,C,False,True,ALM_HP_LOWFLOW,14.0,2026-07-30,2026-07-30
397,TH-02398,C,False,True,ALM_HP_LOWFLOW,14.0,2026-07-30,2026-07-30


## 4. Route attention signals to the right operational queue

The two validated attention signals imply different next actions.

### Technician review

Used when the unit continues reporting but shows a persistent, uncommon OEM-specific signal.

Current example:

- recurrent `ALM_HP_LOWFLOW` on OEM C

The signal is observable and persistent, but its technical meaning remains undocumented.

### Data / connectivity review

Used when the cockpit can no longer determine the current state of the unit because telemetry stopped.

Current example:

- units whose latest reading is at least two days older than the end of the extract

These should not automatically trigger a technician visit because the available evidence may indicate an integration or telemetry problem rather than equipment failure.

In [9]:
attention["review_queue"] = "no_immediate_action"

attention.loc[
    attention["signal_recurrent_error"],
    "review_queue"
] = "technician_review"

attention.loc[
    attention["signal_stale_telemetry"],
    "review_queue"
] = "data_connectivity_review"

### 4.1 Preserve simultaneous signals

A unit may eventually satisfy multiple attention conditions.

Therefore routing should not overwrite the underlying evidence.

The queue represents the recommended first owner of the investigation, while individual signal columns remain available for inspection.

In [10]:
attention["attention_reasons"] = ""

attention.loc[
    attention["signal_stale_telemetry"],
    "attention_reasons"
] += "Telemetry stopped; "

attention.loc[
    attention["signal_recurrent_error"],
    "attention_reasons"
] += (
    "Persistent uncommon OEM signal: "
    + attention["raw_error_signal"].fillna("")
    + "; "
)

attention["attention_reasons"] = (
    attention["attention_reasons"]
    .str.rstrip("; ")
)

## 5. Add service context without turning it into an unsupported score

Operational context can influence how the service organisation handles a unit, but the case does not define SLA or priority rules for service tiers.

We therefore show service context alongside the technical signal without assigning numerical priority weights.

Relevant context includes:

- customer
- postcode region
- service tier
- commissioning date
- last service visit

In [11]:
attention = attention.merge(
    unit_base[
        [
            "unit_id",
            "customer_name",
            "postcode_region",
            "service_tier",
            "commissioning_date",
            "last_service_visit"
        ]
    ],
    on="unit_id",
    how="left"
)

In [12]:
attention_candidates = (
    attention[
        attention["review_queue"] != "no_immediate_action"
    ]
    .sort_values(
        [
            "review_queue",
            "recurrent_error_days",
            "days_since_last_reading"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)

display(
    attention_candidates[
        [
            "unit_id",
            "vendor_canonical",
            "review_queue",
            "attention_reasons",
            "customer_name",
            "postcode_region",
            "service_tier",
            "last_service_visit",
            "latest_reading"
        ]
    ]
)

,unit_id,vendor_canonical,review_queue,attention_reasons,customer_name,postcode_region,service_tier,last_service_visit,latest_reading
22,TH-02023,A,data_connectivity_review,Telemetry stopped,Kerstin Gebhardt,44,optimize,2026-06-02,2026-07-20
279,TH-02280,B,data_connectivity_review,Telemetry stopped,Volker Pfefferle,90,free,2026-01-28,2026-07-20
303,TH-02304,C,data_connectivity_review,Telemetry stopped,Rainer Ahrens,10,free,2025-04-18,2026-07-20
297,TH-02298,C,technician_review,Persistent uncommon OEM signal: ALM_HP_LOWFLOW,Dieter Niehaus,80,free,NaT,2026-07-30
311,TH-02312,C,technician_review,Persistent uncommon OEM signal: ALM_HP_LOWFLOW,Thorsten Oestreich,50,care_plus,2024-09-07,2026-07-30
394,TH-02395,C,technician_review,Persistent uncommon OEM signal: ALM_HP_LOWFLOW,Dagmar Pfefferle,81,optimize,NaT,2026-07-30
397,TH-02398,C,technician_review,Persistent uncommon OEM signal: ALM_HP_LOWFLOW,Dagmar Bartels,90,care_plus,NaT,2026-07-30


## 6. Tomorrow-morning review list

The available evidence supports two operational lists rather than one undifferentiated anomaly queue.

### Technician review

The four OEM C units with persistent `ALM_HP_LOWFLOW` are the strongest candidates for technician review:

- `TH-02298`
- `TH-02312`
- `TH-02395`
- `TH-02398`

Reason:

- the same uncommon raw signal occurs on 14 reporting days for each unit
- only 5.1% of reporting OEM C units show the signal
- the signal remains present through the final day of the extract

Confidence: **medium**

The persistence and rarity are well supported, but the technical meaning of the OEM code is not documented.

### Data / connectivity review

The following units should be investigated separately:

- `TH-02023`
- `TH-02280`
- `TH-02304`

Reason:

- all three are marked connected
- all three previously supplied telemetry
- all three stop reporting after 20 July
- all three are ten days stale at the end of the extract

Confidence: **high** that telemetry stopped; **low** that the heat pump itself requires a technician visit.

The synchronized stop date across three OEMs suggests checking the telemetry/integration path before dispatching field service.

## 7. Export attention list for the prototype

The prototype should consume explicit attention signals and routing rather than reproducing analytical logic inside the UI.

This keeps the product layer simple and makes the prioritization logic auditable.

In [13]:
ATTENTION_OUTPUT = OUTPUT_DIR / "attention_list.csv"

attention_candidates.to_csv(
    ATTENTION_OUTPUT,
    index=False
)

print("Exported:", ATTENTION_OUTPUT)

Exported: ../outputs/attention_list.csv
